# 11 — Modelo de Risco Geográfico (RQ11)

> **Pergunta:** Quais municípios têm maior risco de maus resultados em litíase renal, controlando pelo perfil dos pacientes?
>
> O modelo separa o risco inerente do paciente (idade, sexo, urgência, diagnóstico) do desempenho do sistema de saúde local.
> Municípios com resultados piores do que o esperado para seu perfil de pacientes representam falhas sistêmicas — não população mais doente.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
sys.path.insert(0, ".")
from shared import (
    load_kidney, load_cnes_enriched, load_cnes_names,
    setup_plot_style, save_plot, save_metrics,
    COLORS, DATA_DIR, PLOT_DIR, CITY_NAMES,
)

setup_plot_style()
kidney = load_kidney()
cnes = load_cnes_enriched()
names_df = load_cnes_names()

kidney["sub_diag"] = kidney["DIAG_PRINC"].astype(str).str[:4]
kidney["has_comorbidity"] = (
    kidney["DIAG_SECUN"].notna()
    & (kidney["DIAG_SECUN"].astype(str).str.strip() != "")
).astype(int)
kidney["bed_specialty"] = (
    kidney["ESPEC"].astype(str).map({"01": "surgical", "03": "clinical"}).fillna("other")
)
kidney["month"] = (
    pd.to_datetime(kidney["DT_INTER"], format="%Y%m%d", errors="coerce")
    .dt.month.fillna(6).astype(int)
)
kidney["long_stay"] = (kidney["DIAS_PERM"] > 7).astype(int)

try:
    import requests
    resp = requests.get(
        "https://servicodados.ibge.gov.br/api/v1/localidades/estados/35/municipios",
        timeout=10,
    )
    ibge_names = {str(m["id"])[:6]: m["nome"] for m in resp.json()}
    print(f"Loaded {len(ibge_names)} city names from IBGE API")
except Exception:
    ibge_names = CITY_NAMES
    print(f"Using {len(ibge_names)} city names from local dict")

print(f"Loaded {len(kidney):,} admissions from {kidney['MUNIC_RES'].nunique()} municipalities")
print(f"Long stay rate: {kidney['long_stay'].mean():.1%}")
print(f"Mortality rate: {kidney['MORTE'].mean():.2%}")

Loaded 645 city names from IBGE API
Loaded 206,500 admissions from 908 municipalities
Long stay rate: 4.5%
Mortality rate: 0.35%


## 1. Modelo de Risco do Paciente

Modelo LightGBM treinado **apenas com características do paciente** — sem informação do hospital.
O objetivo é capturar o risco inerente: dado o perfil deste paciente, qual o resultado esperado sob cuidado "médio"?

**Características:** idade, sexo, tipo de admissão (urgência/eletiva), sub-diagnóstico (N200–N209),
presença de comorbidade, especialidade do leito, ano, mês.

**Alvos:** tempo de permanência (regressão), longa permanência >7d (classificação), mortalidade (classificação).

**Validação:** 5-fold cross-validation com predições out-of-sample para toda a base.

In [2]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, r2_score, mean_absolute_error, roc_curve
from sklearn.isotonic import IsotonicRegression
import lightgbm as lgb

FEATURES = ["age", "is_male", "is_emergency", "has_comorbidity", "year", "month"]
CAT_FEATURES = ["sub_diag", "bed_specialty"]

df = kidney.copy()
for col in CAT_FEATURES:
    dummies = pd.get_dummies(df[col], prefix=col, drop_first=True)
    df = pd.concat([df, dummies], axis=1)

feature_cols = FEATURES + [
    c for c in df.columns
    if c.startswith("sub_diag_") or c.startswith("bed_specialty_")
]
X = df[feature_cols].values.astype(np.float32)
y_los = df["DIAS_PERM"].values.astype(np.float32)
y_long = df["long_stay"].values.astype(np.int32)
y_mort = df["MORTE"].values.astype(np.int32)

print(f"Feature matrix: {X.shape[0]:,} patients \u00d7 {X.shape[1]} features")
print(f"Features: {feature_cols}")

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

los_params = dict(
    objective="regression", metric="mae", n_estimators=300,
    max_depth=6, learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, verbose=-1, random_state=42,
)
cls_params = dict(
    objective="binary", metric="auc", n_estimators=300,
    max_depth=6, learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, verbose=-1, random_state=42,
    is_unbalance=True,
)

pred_los = np.zeros(len(X))
pred_long = np.zeros(len(X))
pred_mort = np.zeros(len(X))

for fold, (train_idx, test_idx) in enumerate(kf.split(X, y_long)):
    rng_fold = np.random.RandomState(42 + fold)
    perm = rng_fold.permutation(len(train_idx))
    cal_size = int(len(train_idx) * 0.2)
    cal_idx = train_idx[perm[:cal_size]]
    fit_idx = train_idx[perm[cal_size:]]

    m_los = lgb.LGBMRegressor(**los_params)
    m_los.fit(X[train_idx], y_los[train_idx])
    pred_los[test_idx] = m_los.predict(X[test_idx])

    m_long = lgb.LGBMClassifier(**cls_params)
    m_long.fit(X[fit_idx], y_long[fit_idx])
    cal_p = m_long.predict_proba(X[cal_idx])[:, 1]
    iso_l = IsotonicRegression(out_of_bounds="clip")
    iso_l.fit(cal_p, y_long[cal_idx])
    pred_long[test_idx] = iso_l.transform(m_long.predict_proba(X[test_idx])[:, 1])

    m_mort = lgb.LGBMClassifier(**cls_params)
    m_mort.fit(X[fit_idx], y_mort[fit_idx])
    cal_m = m_mort.predict_proba(X[cal_idx])[:, 1]
    iso_m = IsotonicRegression(out_of_bounds="clip")
    iso_m.fit(cal_m, y_mort[cal_idx])
    pred_mort[test_idx] = iso_m.transform(m_mort.predict_proba(X[test_idx])[:, 1])

    print(f"  Fold {fold+1}/5 done")

r2 = r2_score(y_los, pred_los)
mae = mean_absolute_error(y_los, pred_los)
auc_long = roc_auc_score(y_long, pred_long)
auc_mort = roc_auc_score(y_mort, pred_mort)

print(f"\nLOS Regression:           R\u00b2={r2:.3f}  MAE={mae:.2f}d")
print(f"Long Stay (calibrated):   AUC={auc_long:.3f}")
print(f"Mortality (calibrated):   AUC={auc_mort:.3f}")

Feature matrix: 206,500 patients × 12 features
Features: ['age', 'is_male', 'is_emergency', 'has_comorbidity', 'year', 'month', 'sub_diag_N200', 'sub_diag_N201', 'sub_diag_N202', 'sub_diag_N209', 'bed_specialty_other', 'bed_specialty_surgical']


/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fold 1/5 done


...

  Fold 4/5 done


/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fold 5/5 done

LOS Regression:           R²=0.077  MAE=1.62d
Long Stay (calibrated):   AUC=0.704
Mortality (calibrated):   AUC=0.725


In [3]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
idx = np.random.RandomState(42).choice(len(pred_los), min(5000, len(pred_los)), replace=False)
ax.scatter(pred_los[idx], y_los[idx], alpha=0.1, s=5, color=COLORS["primary"])
ax.plot([0, 20], [0, 20], "r--", alpha=0.5)
ax.set_xlabel("LOS Previsto (dias)")
ax.set_ylabel("LOS Real (dias)")
ax.set_title(f"LOS: R\u00b2={r2:.3f}, MAE={mae:.2f}d")
ax.set_xlim(0, 20)
ax.set_ylim(0, 20)

fpr, tpr, _ = roc_curve(y_long, pred_long)
ax = axes[1]
ax.plot(fpr, tpr, color=COLORS["primary"], lw=2)
ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax.set_xlabel("Taxa Falso Positivo")
ax.set_ylabel("Taxa Verdadeiro Positivo")
ax.set_title(f"Longa Perman\u00eancia: AUC={auc_long:.3f}")
ax.set_aspect("equal")

fpr_m, tpr_m, _ = roc_curve(y_mort, pred_mort)
ax = axes[2]
ax.plot(fpr_m, tpr_m, color=COLORS["danger"], lw=2)
ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax.set_xlabel("Taxa Falso Positivo")
ax.set_ylabel("Taxa Verdadeiro Positivo")
ax.set_title(f"Mortalidade: AUC={auc_mort:.3f}")
ax.set_aspect("equal")

fig.suptitle("Desempenho do Modelo de Risco (apenas caracter\u00edsticas do paciente)",
             fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
save_plot(fig, "model_performance", prefix="11")

  Saved: 11_model_performance.png


## 2. Interpreta\u00e7\u00e3o do Modelo \u2014 SHAP

Quais caracter\u00edsticas do paciente mais influenciam o risco previsto?

In [4]:
import shap

final_los = lgb.LGBMRegressor(**los_params)
final_los.fit(X, y_los)

explainer = shap.TreeExplainer(final_los)
sample_idx = np.random.RandomState(42).choice(len(X), min(5000, len(X)), replace=False)
shap_values = explainer.shap_values(X[sample_idx])

fig, ax = plt.subplots(figsize=(10, 6))
importance = np.abs(shap_values).mean(axis=0)
order = np.argsort(importance)
ax.barh(range(len(order)), importance[order], color=COLORS["primary"])
ax.set_yticks(range(len(order)))
ax.set_yticklabels([feature_cols[i] for i in order])
ax.set_title("Import\u00e2ncia SHAP \u2014 Fatores de Risco do Paciente (LOS)",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Impacto m\u00e9dio no LOS previsto (dias)")
fig.tight_layout()
save_plot(fig, "shap_risk_factors", prefix="11")

fig = plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X[sample_idx], feature_names=feature_cols,
                  show=False, max_display=12)
plt.title("SHAP Beeswarm \u2014 Risco do Paciente", fontsize=14, fontweight="bold")
plt.tight_layout()
save_plot(plt.gcf(), "shap_beeswarm", prefix="11")

/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  Saved: 11_shap_risk_factors.png


  Saved: 11_shap_beeswarm.png


## 3. Agrega\u00e7\u00e3o por Munic\u00edpio de Resid\u00eancia

Predi\u00e7\u00f5es out-of-sample agregadas por munic\u00edpio. O **gap** entre resultado real e previsto revela onde o sistema
de sa\u00fade funciona melhor ou pior do que o esperado para o perfil de pacientes daquela cidade.

- **Gap positivo** \u2192 resultado pior que o esperado \u2192 sistema est\u00e1 falhando
- **Gap negativo** \u2192 resultado melhor que o esperado \u2192 sistema funciona bem

In [5]:
df["pred_los"] = pred_los
df["pred_long"] = pred_long
df["pred_mort"] = pred_mort

MIN_ADMISSIONS = 30

city_agg = df.groupby("MUNIC_RES").agg(
    n=("DIAS_PERM", "count"),
    actual_los=("DIAS_PERM", "mean"),
    actual_long_rate=("long_stay", "mean"),
    actual_mort_rate=("MORTE", "mean"),
    actual_cost=("VAL_TOT", "mean"),
    pred_los=("pred_los", "mean"),
    pred_long_rate=("pred_long", "mean"),
    pred_mort_rate=("pred_mort", "mean"),
    mean_age=("age", "mean"),
    pct_male=("is_male", "mean"),
    pct_emergency=("is_emergency", "mean"),
    pct_comorbidity=("has_comorbidity", "mean"),
    migration_rate=("migrated", "mean"),
).reset_index()

city_agg = city_agg[city_agg["n"] >= MIN_ADMISSIONS].copy()

city_agg["gap_los"] = city_agg["actual_los"] - city_agg["pred_los"]
city_agg["gap_long"] = city_agg["actual_long_rate"] - city_agg["pred_long_rate"]
city_agg["gap_mort"] = city_agg["actual_mort_rate"] - city_agg["pred_mort_rate"]

for col in ["gap_los", "gap_long", "gap_mort"]:
    std = city_agg[col].std()
    city_agg[f"{col}_z"] = (city_agg[col] - city_agg[col].mean()) / std if std > 0 else 0

city_agg["gap_composite"] = (
    city_agg["gap_los_z"] * 0.4
    + city_agg["gap_long_z"] * 0.3
    + city_agg["gap_mort_z"] * 0.3
)

city_agg["city_name"] = city_agg["MUNIC_RES"].map(ibge_names).fillna(city_agg["MUNIC_RES"])
city_agg = city_agg.sort_values("gap_composite", ascending=False)

print(f"Munic\u00edpios com \u2265{MIN_ADMISSIONS} interna\u00e7\u00f5es: {len(city_agg)}")
cols = ["city_name", "n", "actual_los", "pred_los", "gap_los", "gap_composite"]
print(f"\nTop 5 \u2014 piores gaps (sistema falha mais):")
print(city_agg[cols].head().to_string(index=False))
print(f"\nTop 5 \u2014 melhores gaps (sistema funciona bem):")
print(city_agg[cols].tail().to_string(index=False))

Municípios com ≥30 internações: 489

Top 5 — piores gaps (sistema falha mais):
   city_name  n  actual_los  pred_los  gap_los  gap_composite
Araçariguama 37    4.486486  2.391614 2.094873       4.026200
Paranapanema 55    5.945455  2.854111 3.091343       3.758158
   Rinópolis 39    3.435897  2.588398 0.847500       3.416531
       Iperó 97    4.494845  2.407754 2.087091       2.592486
     Sarapuí 33    4.575758  2.197718 2.378039       2.404767

Top 5 — melhores gaps (sistema funciona bem):
        city_name   n  actual_los  pred_los   gap_los  gap_composite
            Guará 147    1.367347  2.780826 -1.413479      -1.474603
         Orlândia 318    1.267296  2.892511 -1.625216      -1.499724
Cristais Paulista  51    0.764706  2.426731 -1.662025      -1.556580
        Buritizal  80    1.412500  2.935933 -1.523433      -1.703433
        Auriflama  30    1.900000  3.503589 -1.603589      -2.017769


In [6]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

panels = [
    ("pred_los", "actual_los", "gap_los", "Tempo de Perman\u00eancia", "dias"),
    ("pred_long_rate", "actual_long_rate", "gap_long", "Taxa Longa Perman\u00eancia", "propor\u00e7\u00e3o"),
    ("pred_mort_rate", "actual_mort_rate", "gap_mort", "Taxa de Mortalidade", "propor\u00e7\u00e3o"),
]

for ax, (pred_col, actual_col, gap_col, title, unit) in zip(axes, panels):
    sizes = np.clip(city_agg["n"] / 10, 10, 200)
    scatter = ax.scatter(
        city_agg[pred_col], city_agg[actual_col],
        c=city_agg[gap_col], cmap="RdYlGn_r", s=sizes,
        alpha=0.7, edgecolors="white", linewidths=0.5,
    )
    lim = max(city_agg[pred_col].max(), city_agg[actual_col].max()) * 1.1
    ax.plot([0, lim], [0, lim], "k--", alpha=0.3)
    ax.set_xlabel(f"Previsto ({unit})")
    ax.set_ylabel(f"Real ({unit})")
    ax.set_title(title, fontweight="bold")
    plt.colorbar(scatter, ax=ax, label=f"Gap ({unit})", shrink=0.8)

    top3 = city_agg.nlargest(3, gap_col)
    for _, row in top3.iterrows():
        ax.annotate(row["city_name"], (row[pred_col], row[actual_col]),
                    fontsize=7, alpha=0.8, ha="left",
                    xytext=(5, 5), textcoords="offset points")

fig.suptitle("Risco Previsto vs Resultado Real por Munic\u00edpio\n"
             "(acima da diagonal = sistema falha; abaixo = sistema funciona bem)",
             fontsize=14, fontweight="bold", y=1.05)
fig.tight_layout()
save_plot(fig, "risk_gap_scatter", prefix="11")

  Saved: 11_risk_gap_scatter.png


## 4. Scorecard Municipal

Ranking dos munic\u00edpios por gap composto (combina\u00e7\u00e3o ponderada de gap de LOS, longa perman\u00eancia e mortalidade).
O gap composto \u00e9 normalizado (z-score) para comparabilidade entre as tr\u00eas dimens\u00f5es.

In [7]:
n_show = 15

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

worst = city_agg.nlargest(n_show, "gap_composite")
ax = axes[0]
ax.barh(range(len(worst)), worst["gap_composite"].values,
        color=[COLORS["danger"] if g > 0 else COLORS["muted"]
               for g in worst["gap_composite"]])
ax.set_yticks(range(len(worst)))
ax.set_yticklabels([f"{r['city_name']} (n={r['n']:,})"
                    for _, r in worst.iterrows()], fontsize=9)
ax.set_xlabel("Gap Composto (z-score)")
ax.set_title("Pior Desempenho\n(resultado pior que o esperado)", fontweight="bold")
ax.invert_yaxis()

best = city_agg.nsmallest(n_show, "gap_composite")
ax = axes[1]
ax.barh(range(len(best)), best["gap_composite"].values,
        color=[COLORS["success"] if g < 0 else COLORS["muted"]
               for g in best["gap_composite"]])
ax.set_yticks(range(len(best)))
ax.set_yticklabels([f"{r['city_name']} (n={r['n']:,})"
                    for _, r in best.iterrows()], fontsize=9)
ax.set_xlabel("Gap Composto (z-score)")
ax.set_title("Melhor Desempenho\n(resultado melhor que o esperado)", fontweight="bold")
ax.invert_yaxis()

fig.suptitle("Scorecard Municipal \u2014 Desempenho Ajustado por Risco",
             fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
save_plot(fig, "city_scorecard", prefix="11")

print("\n=== TOP 15 PIORES ===")
print(worst[["city_name", "n", "actual_los", "pred_los", "gap_los",
             "actual_mort_rate", "pred_mort_rate", "gap_mort",
             "gap_composite"]].to_string(index=False, float_format="%.3f"))

print("\n=== TOP 15 MELHORES ===")
print(best[["city_name", "n", "actual_los", "pred_los", "gap_los",
            "actual_mort_rate", "pred_mort_rate", "gap_mort",
            "gap_composite"]].to_string(index=False, float_format="%.3f"))

  Saved: 11_city_scorecard.png

=== TOP 15 PIORES ===
     city_name   n  actual_los  pred_los  gap_los  actual_mort_rate  pred_mort_rate  gap_mort  gap_composite
  Araçariguama  37       4.486     2.392    2.095             0.054           0.003     0.051          4.026
  Paranapanema  55       5.945     2.854    3.091             0.000           0.005    -0.005          3.758
     Rinópolis  39       3.436     2.588    0.847             0.051           0.003     0.048          3.417
         Iperó  97       4.495     2.408    2.087             0.010           0.003     0.007          2.592
       Sarapuí  33       4.576     2.198    2.378             0.000           0.003    -0.003          2.405
 Cesário Lange  41       4.049     2.668    1.381             0.024           0.003     0.021          2.152
       Parapuã  37       3.568     2.484    1.084             0.027           0.004     0.023          2.136
  Capão Bonito 116       3.422     2.265    1.158             0.017       

## 5. Fatores Associados ao Gap de Desempenho

O que distingue munic\u00edpios com pior e melhor desempenho ajustado por risco?

In [8]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

correlates = [
    ("migration_rate", "Taxa de Migra\u00e7\u00e3o", COLORS["primary"]),
    ("mean_age", "Idade M\u00e9dia", COLORS["secondary"]),
    ("pct_emergency", "% Urg\u00eancia", COLORS["warning"]),
    ("n", "Volume de Interna\u00e7\u00f5es", COLORS["success"]),
]

for ax, (col, label, color) in zip(axes.flat, correlates):
    sizes = np.clip(city_agg["n"] / 5, 10, 100)
    ax.scatter(city_agg[col], city_agg["gap_composite"], s=sizes, alpha=0.5, color=color)
    r, p = stats.spearmanr(city_agg[col], city_agg["gap_composite"])
    ax.set_xlabel(label)
    ax.set_ylabel("Gap Composto")
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    ax.set_title(f"{label} vs Gap (\u03c1={r:.2f}, p={p:.3f} {sig})", fontweight="bold")
    ax.axhline(0, color="gray", linestyle="--", alpha=0.3)

fig.suptitle("Correlatos do Gap de Desempenho Municipal",
             fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
save_plot(fig, "gap_correlates", prefix="11")

print("\nCorrela\u00e7\u00f5es (Spearman) com gap composto:")
for col, label, _ in correlates:
    r, p = stats.spearmanr(city_agg[col], city_agg["gap_composite"])
    print(f"  {label:25s}: \u03c1={r:+.3f}  p={p:.4f}")

  Saved: 11_gap_correlates.png

Correlações (Spearman) com gap composto:
  Taxa de Migração         : ρ=-0.057  p=0.2061
  Idade Média              : ρ=-0.084  p=0.0630
  % Urgência               : ρ=-0.173  p=0.0001
  Volume de Internações    : ρ=+0.052  p=0.2495


## 6. Cruzamento com Infraestrutura Hospitalar

O gap de desempenho est\u00e1 associado \u00e0 qualidade dos hospitais utilizados pelos pacientes de cada munic\u00edpio?

In [9]:
hosp_stats = df.groupby("CNES").agg(
    h_los=("DIAS_PERM", "mean"),
    h_mort=("MORTE", "mean"),
    h_n=("DIAS_PERM", "count"),
).reset_index()
hosp_stats = hosp_stats[hosp_stats["h_n"] >= 10]

patient_hosp = df.merge(hosp_stats[["CNES", "h_los", "h_mort"]], on="CNES", how="left")

city_hosp_quality = patient_hosp.groupby("MUNIC_RES").agg(
    n_hospitals_used=("CNES", "nunique"),
    mean_hosp_los=("h_los", "mean"),
    mean_hosp_mort=("h_mort", "mean"),
).reset_index()

city_full = city_agg.merge(city_hosp_quality, on="MUNIC_RES", how="left")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
valid = city_full.dropna(subset=["n_hospitals_used"])
ax.scatter(valid["n_hospitals_used"], valid["gap_composite"],
           s=valid["n"]/5, alpha=0.5, color=COLORS["primary"])
r, p = stats.spearmanr(valid["n_hospitals_used"], valid["gap_composite"])
ax.set_xlabel("N\u00ba de Hospitais Utilizados")
ax.set_ylabel("Gap Composto")
ax.set_title(f"Diversidade Hospitalar vs Gap\n(\u03c1={r:.2f}, p={p:.3f})", fontweight="bold")
ax.axhline(0, color="gray", linestyle="--", alpha=0.3)

ax = axes[1]
valid2 = city_full.dropna(subset=["mean_hosp_los"])
ax.scatter(valid2["mean_hosp_los"], valid2["gap_composite"],
           s=valid2["n"]/5, alpha=0.5, color=COLORS["danger"])
rho2, p2 = stats.spearmanr(valid2["mean_hosp_los"], valid2["gap_composite"])
ax.set_xlabel("LOS M\u00e9dio dos Hospitais")
ax.set_ylabel("Gap Composto")
ax.set_title(f"Qualidade Hospitalar (LOS) vs Gap\n(\u03c1={rho2:.2f}, p={p2:.3f})", fontweight="bold")
ax.axhline(0, color="gray", linestyle="--", alpha=0.3)

ax = axes[2]
valid3 = city_full.dropna(subset=["mean_hosp_mort"])
ax.scatter(valid3["mean_hosp_mort"] * 100, valid3["gap_composite"],
           s=valid3["n"]/5, alpha=0.5, color=COLORS["warning"])
r3, p3 = stats.spearmanr(valid3["mean_hosp_mort"], valid3["gap_composite"])
ax.set_xlabel("Mortalidade M\u00e9dia dos Hospitais (%)")
ax.set_ylabel("Gap Composto")
ax.set_title(f"Qualidade Hospitalar (Mortalidade) vs Gap\n(\u03c1={r3:.2f}, p={p3:.3f})", fontweight="bold")
ax.axhline(0, color="gray", linestyle="--", alpha=0.3)

fig.suptitle("Infraestrutura Hospitalar e Gap de Desempenho",
             fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
save_plot(fig, "infrastructure_gap", prefix="11")

print("\nCorrela\u00e7\u00f5es infraestrutura vs gap:")
print(f"  N\u00ba hospitais:    \u03c1={r:+.3f}  p={p:.4f}")
print(f"  LOS hospitalar:  \u03c1={rho2:+.3f}  p={p2:.4f}")
print(f"  Mort hospitalar: \u03c1={r3:+.3f}  p={p3:.4f}")

  Saved: 11_infrastructure_gap.png

Correlações infraestrutura vs gap:
  Nº hospitais:    ρ=+0.292  p=0.0000
  LOS hospitalar:  ρ=+0.528  p=0.0000
  Mort hospitalar: ρ=+0.425  p=0.0000


## 7. Validação Rigorosa do Modelo

O modelo precisa passar por validação real antes de qualquer conclusão:
calibração, validação temporal, significância estatística dos gaps,
análise por subgrupo, e cruzamento com dados independentes.

In [10]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
prob_true, prob_pred = calibration_curve(y_long, pred_long, n_bins=10, strategy="quantile")
ax.plot(prob_pred, prob_true, "bo-", label="Modelo", lw=2)
ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Calibração perfeita")
brier_long = brier_score_loss(y_long, pred_long)
ax.set_title(f"Calibração — Longa Permanência\nBrier={brier_long:.4f} (base rate={y_long.mean():.3f})",
             fontweight="bold")
ax.set_xlabel("Probabilidade Prevista")
ax.set_ylabel("Proporção Real")
ax.legend()
ax.set_aspect("equal")

ax = axes[1]
prob_true_m, prob_pred_m = calibration_curve(y_mort, pred_mort, n_bins=10, strategy="quantile")
ax.plot(prob_pred_m, prob_true_m, "ro-", label="Modelo", lw=2)
ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Calibração perfeita")
brier_mort = brier_score_loss(y_mort, pred_mort)
ax.set_title(f"Calibração — Mortalidade\nBrier={brier_mort:.4f} (base rate={y_mort.mean():.4f})",
             fontweight="bold")
ax.set_xlabel("Probabilidade Prevista")
ax.set_ylabel("Proporção Real")
ax.legend()
ax.set_aspect("equal")

fig.suptitle("Calibração do Modelo: Probabilidades Previstas vs Reais",
             fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
save_plot(fig, "calibration", prefix="11")

print(f"Brier Score (longa permanência): {brier_long:.4f}")
print(f"  Base rate: {y_long.mean():.4f}")
print(f"  Skill vs base rate: {1 - brier_long / (y_long.mean() * (1 - y_long.mean())):.3f}")
print(f"Brier Score (mortalidade):       {brier_mort:.4f}")
print(f"  Base rate: {y_mort.mean():.4f}")
print(f"  Skill vs base rate: {1 - brier_mort / (y_mort.mean() * (1 - y_mort.mean())):.3f}")

  Saved: 11_calibration.png
Brier Score (longa permanência): 0.0420
  Base rate: 0.0452
  Skill vs base rate: 0.027
Brier Score (mortalidade):       0.0034
  Base rate: 0.0035
  Skill vs base rate: 0.001


### 7.1 Validação Temporal

Treino em 2016–2023, teste em 2024–2025. Se o modelo generaliza no tempo,
as métricas no teste temporal devem ser próximas das do CV.

In [11]:
mask_train = df["year"] <= 2023
mask_test = df["year"] >= 2024

X_train_t = X[mask_train.values]
X_test_t = X[mask_test.values]
y_los_train_t = y_los[mask_train.values]
y_los_test_t = y_los[mask_test.values]
y_long_train_t = y_long[mask_train.values]
y_long_test_t = y_long[mask_test.values]
y_mort_train_t = y_mort[mask_train.values]
y_mort_test_t = y_mort[mask_test.values]

rng_t = np.random.RandomState(99)
perm_t = rng_t.permutation(len(X_train_t))
cal_sz = int(len(X_train_t) * 0.2)
cal_t = perm_t[:cal_sz]
fit_t = perm_t[cal_sz:]

mt_los = lgb.LGBMRegressor(**los_params)
mt_los.fit(X_train_t, y_los_train_t)
pt_los = mt_los.predict(X_test_t)

mt_long = lgb.LGBMClassifier(**cls_params)
mt_long.fit(X_train_t[fit_t], y_long_train_t[fit_t])
iso_tl = IsotonicRegression(out_of_bounds="clip")
iso_tl.fit(mt_long.predict_proba(X_train_t[cal_t])[:, 1], y_long_train_t[cal_t])
pt_long = iso_tl.transform(mt_long.predict_proba(X_test_t)[:, 1])

mt_mort = lgb.LGBMClassifier(**cls_params)
mt_mort.fit(X_train_t[fit_t], y_mort_train_t[fit_t])
iso_tm = IsotonicRegression(out_of_bounds="clip")
iso_tm.fit(mt_mort.predict_proba(X_train_t[cal_t])[:, 1], y_mort_train_t[cal_t])
pt_mort = iso_tm.transform(mt_mort.predict_proba(X_test_t)[:, 1])

r2_t = r2_score(y_los_test_t, pt_los)
mae_t = mean_absolute_error(y_los_test_t, pt_los)
auc_long_t = roc_auc_score(y_long_test_t, pt_long)
auc_mort_t = roc_auc_score(y_mort_test_t, pt_mort)

print("Validação Temporal (treino ≤2023, teste ≥2024)")
print(f"  Treino: {mask_train.sum():,} pacientes")
print(f"  Teste:  {mask_test.sum():,} pacientes")
print(f"\n{'Métrica':25s} {'CV 5-fold':>10} {'Temporal':>10} {'Delta':>10}")
print("-" * 58)
print(f"{'LOS R²':25s} {r2:>10.3f} {r2_t:>10.3f} {r2_t - r2:>+10.3f}")
print(f"{'LOS MAE (dias)':25s} {mae:>10.2f} {mae_t:>10.2f} {mae_t - mae:>+10.2f}")
print(f"{'Long Stay AUC':25s} {auc_long:>10.3f} {auc_long_t:>10.3f} {auc_long_t - auc_long:>+10.3f}")
print(f"{'Mortalidade AUC':25s} {auc_mort:>10.3f} {auc_mort_t:>10.3f} {auc_mort_t - auc_mort:>+10.3f}")

temporal_stable = abs(r2_t - r2) < 0.05 and abs(auc_long_t - auc_long) < 0.05
print(f"\nEstabilidade temporal: {'PASS' if temporal_stable else 'FAIL'}")
print(f"  Critério: delta < 0.05 em R² e AUC")

/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Validação Temporal (treino ≤2023, teste ≥2024)
  Treino: 144,153 pacientes
  Teste:  62,347 pacientes

Métrica                    CV 5-fold   Temporal      Delta
----------------------------------------------------------
LOS R²                         0.077      0.076     -0.001
LOS MAE (dias)                  1.62       1.56      -0.06
Long Stay AUC                  0.704      0.714     +0.010
Mortalidade AUC                0.725      0.734     +0.009

Estabilidade temporal: PASS
  Critério: delta < 0.05 em R² e AUC


## 8. Significância Estatística dos Gaps Municipais

Nem todo gap é real. Municípios com poucos pacientes podem ter gaps grandes por acaso.
Bootstrap 95% CI para cada município: se o intervalo não inclui zero, o gap é estatisticamente significativo.

In [12]:
N_BOOTSTRAP = 1000
rng = np.random.RandomState(42)

ci_results = []
for _, row in city_agg.iterrows():
    code = row["MUNIC_RES"]
    mask = df["MUNIC_RES"] == code
    residuals = df.loc[mask, "DIAS_PERM"].values - df.loc[mask, "pred_los"].values
    n = len(residuals)

    boot_means = np.array([
        rng.choice(residuals, size=n, replace=True).mean()
        for _ in range(N_BOOTSTRAP)
    ])

    ci_low = np.percentile(boot_means, 2.5)
    ci_high = np.percentile(boot_means, 97.5)
    significant = (ci_low > 0) or (ci_high < 0)

    ci_results.append({
        "MUNIC_RES": code,
        "gap_los_ci_low": ci_low,
        "gap_los_ci_high": ci_high,
        "gap_significant": significant,
    })

ci_df = pd.DataFrame(ci_results)
city_agg = city_agg.merge(ci_df, on="MUNIC_RES", how="left")

n_sig = city_agg["gap_significant"].sum()
n_sig_pos = ((city_agg["gap_significant"]) & (city_agg["gap_los"] > 0)).sum()
n_sig_neg = ((city_agg["gap_significant"]) & (city_agg["gap_los"] < 0)).sum()

print(f"Gaps significativos (95% CI exclui zero): {n_sig}/{len(city_agg)} ({n_sig/len(city_agg):.0%})")
print(f"  Significativamente piores: {n_sig_pos}")
print(f"  Significativamente melhores: {n_sig_neg}")
print(f"  Não significativos: {len(city_agg) - n_sig}")

fig, ax = plt.subplots(figsize=(12, 6))
sorted_cities = city_agg.sort_values("gap_los")
x = range(len(sorted_cities))
colors = ["#DC2626" if s and g > 0 else "#059669" if s and g < 0 else "#D1D5DB"
          for s, g in zip(sorted_cities["gap_significant"], sorted_cities["gap_los"])]
ax.bar(x, sorted_cities["gap_los"].values, color=colors, width=1.0, edgecolor="none")
ax.fill_between(x, sorted_cities["gap_los_ci_low"].values,
                sorted_cities["gap_los_ci_high"].values,
                alpha=0.15, color="gray", label="95% CI")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel(f"Municípios (ordenados por gap, n={len(sorted_cities)})")
ax.set_ylabel("Gap LOS (dias)")
ax.set_title(f"Gap de LOS por Município com 95% CI Bootstrap\n"
             f"Vermelho: significativamente pior ({n_sig_pos}), "
             f"Verde: significativamente melhor ({n_sig_neg}), "
             f"Cinza: não significativo ({len(city_agg) - n_sig})",
             fontweight="bold", fontsize=12)
ax.set_xticks([])
fig.tight_layout()
save_plot(fig, "gap_significance", prefix="11")

Gaps significativos (95% CI exclui zero): 224/489 (46%)
  Significativamente piores: 81
  Significativamente melhores: 143
  Não significativos: 265


  Saved: 11_gap_significance.png


## 9. Avaliação por Subgrupo

O modelo funciona igualmente bem para todos os tipos de paciente?
Se não, os gaps municipais podem refletir viés do modelo, não falha do sistema.

In [13]:
subgroups = [
    ("Idade <40", df["age"] < 40),
    ("Idade 40-60", (df["age"] >= 40) & (df["age"] < 60)),
    ("Idade \u226560", df["age"] >= 60),
    ("Masculino", df["is_male"] == 1),
    ("Feminino", df["is_male"] == 0),
    ("Urg\u00eancia", df["is_emergency"] == 1),
    ("Eletiva", df["is_emergency"] == 0),
    ("N200 (rim)", df["sub_diag"] == "N200"),
    ("N201 (ureter)", df["sub_diag"] == "N201"),
    ("N202 (ambos)", df["sub_diag"] == "N202"),
]

print(f"{'Subgrupo':20s} {'N':>8} {'R\u00b2':>8} {'MAE':>8} {'AUC_LS':>8} {'AUC_M':>8} {'Vi\u00e9s LOS':>10}")
print("-" * 75)

subgroup_results = []
for name, mask in subgroups:
    m = mask.values
    n_sub = m.sum()
    if n_sub < 100:
        continue
    r2_s = r2_score(y_los[m], pred_los[m])
    mae_s = mean_absolute_error(y_los[m], pred_los[m])
    bias = (pred_los[m] - y_los[m]).mean()
    try:
        auc_l = roc_auc_score(y_long[m], pred_long[m])
    except ValueError:
        auc_l = float("nan")
    try:
        auc_m = roc_auc_score(y_mort[m], pred_mort[m])
    except ValueError:
        auc_m = float("nan")
    print(f"{name:20s} {n_sub:>8,} {r2_s:>8.3f} {mae_s:>8.2f} {auc_l:>8.3f} {auc_m:>8.3f} {bias:>+10.2f}")
    subgroup_results.append({"name": name, "n": n_sub, "r2": r2_s, "mae": mae_s,
                             "auc_long": auc_l, "auc_mort": auc_m, "bias": bias})

max_bias = max(abs(r["bias"]) for r in subgroup_results)
print(f"\nMaior vi\u00e9s absoluto: {max_bias:.2f} dias")
print(f"Subgrupo com maior vi\u00e9s: {max((r for r in subgroup_results), key=lambda r: abs(r['bias']))['name']}")
print(f"\nAvalia\u00e7\u00e3o: {'PASS' if max_bias < 0.5 else 'ATEN\u00c7\u00c3O'} (crit\u00e9rio: vi\u00e9s < 0.5 dias)")

Subgrupo                    N       R²      MAE   AUC_LS    AUC_M   Viés LOS
---------------------------------------------------------------------------
Idade <40              70,303    0.062     1.49    0.662    0.573      +0.00
Idade 40-60            93,027    0.068     1.55    0.692    0.640      -0.00
Idade ≥60              43,170    0.091     1.99    0.725    0.651      +0.00
Masculino              97,462    0.071     1.53    0.713    0.715      +0.00
Feminino              109,038    0.080     1.70    0.688    0.734      -0.00
Urgência              116,672    0.036     1.97    0.659    0.724      -0.00


Eletiva                89,828    0.061     1.18    0.632    0.585      +0.00


N200 (rim)             72,093    0.075     1.72    0.696    0.742      -0.00
N201 (ureter)          80,523    0.074     1.51    0.714    0.733      +0.00
N202 (ambos)           39,539    0.085     1.55    0.689    0.688      +0.00

Maior viés absoluto: 0.00 dias
Subgrupo com maior viés: Idade ≥60

Avaliação: PASS (critério: viés < 0.5 dias)


## 10. Validação Cruzada com Eficiência Hospitalar

O teste definitivo: os municípios que o modelo aponta como piores mandam pacientes
a hospitais que outros notebooks já classificaram como ineficientes?

Se sim, o gap captura algo real. Se não, pode ser ruído.

In [14]:
from shared import load_metrics, hospital_name

# Compute hospital efficiency (same method as nb05/nb10)
hp = df.copy()
sys_med_cost = hp.groupby("PROC_REA")["VAL_TOT"].transform("median")
sys_med_los = hp.groupby("PROC_REA")["DIAS_PERM"].transform("median")
hp["cost_ratio"] = hp["VAL_TOT"] / sys_med_cost.replace(0, np.nan)
hp["los_ratio"] = hp["DIAS_PERM"] / sys_med_los.replace(0, np.nan)
hp["resolved"] = ((hp["MORTE"] == 0) & (hp["DIAS_PERM"] <= sys_med_los)).astype(int)

hp_agg = hp.groupby(["CNES", "PROC_REA"]).agg(
    n=("VAL_TOT", "count"),
    mean_cost_ratio=("cost_ratio", "mean"),
    mean_los_ratio=("los_ratio", "mean"),
    mortality=("MORTE", "mean"),
    resolution=("resolved", "mean"),
).reset_index()
hp_agg = hp_agg[hp_agg["n"] >= 5]

hosp_eff = hp_agg.groupby("CNES").apply(
    lambda g: pd.Series({
        "eff_n": g["n"].sum(),
        "w_cost_ratio": np.average(g["mean_cost_ratio"], weights=g["n"]),
        "w_los_ratio": np.average(g["mean_los_ratio"], weights=g["n"]),
        "w_mortality": np.average(g["mortality"], weights=g["n"]),
        "w_resolution": np.average(g["resolution"], weights=g["n"]),
    }),
    include_groups=False,
).reset_index()
hosp_eff["efficiency"] = hosp_eff["w_resolution"] / hosp_eff["w_cost_ratio"].replace(0, np.nan)
hosp_eff = hosp_eff.dropna(subset=["efficiency"])
eff_median = hosp_eff["efficiency"].median()

# For each patient, get their hospital's efficiency
patient_eff = df.merge(hosp_eff[["CNES", "efficiency"]], on="CNES", how="left")
city_eff = patient_eff.groupby("MUNIC_RES").agg(
    mean_eff=("efficiency", "mean"),
    pct_above_median=("efficiency", lambda x: (x > eff_median).mean()),
).reset_index()

city_v = city_agg.merge(city_eff, on="MUNIC_RES", how="left")

# Correlation: gap vs hospital efficiency
valid = city_v.dropna(subset=["mean_eff"])
r_eff, p_eff = stats.spearmanr(valid["mean_eff"], valid["gap_composite"])
print(f"Correla\u00e7\u00e3o gap composto vs efici\u00eancia hospitalar: \u03c1={r_eff:.3f}, p={p_eff:.4f}")
print(f"  Esperado: negativo (hospitais eficientes \u2192 menor gap)")
print(f"  Resultado: {'COERENTE' if r_eff < 0 else 'INCOERENTE'}")

fig, ax = plt.subplots(figsize=(10, 6))
colors_scatter = ["#DC2626" if g > 0 else "#059669" for g in valid["gap_composite"]]
ax.scatter(valid["mean_eff"], valid["gap_composite"],
           s=valid["n"]/5, alpha=0.5, c=colors_scatter, edgecolors="white", linewidths=0.3)
ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
ax.axvline(eff_median, color="gray", linestyle=":", alpha=0.5, label=f"Efici\u00eancia mediana={eff_median:.2f}")

z = np.polyfit(valid["mean_eff"], valid["gap_composite"], 1)
p_fit = np.poly1d(z)
x_line = np.linspace(valid["mean_eff"].min(), valid["mean_eff"].max(), 100)
ax.plot(x_line, p_fit(x_line), "k-", alpha=0.5, lw=2)

ax.set_xlabel("Efici\u00eancia M\u00e9dia dos Hospitais Utilizados")
ax.set_ylabel("Gap Composto")
ax.set_title(f"Valida\u00e7\u00e3o: Gap vs Efici\u00eancia Hospitalar Independente\n"
             f"(\u03c1={r_eff:.3f}, p={p_eff:.4f})",
             fontweight="bold", fontsize=13)
ax.legend()
fig.tight_layout()
save_plot(fig, "gap_vs_efficiency", prefix="11")

Correlação gap composto vs eficiência hospitalar: ρ=-0.346, p=0.0000
  Esperado: negativo (hospitais eficientes → menor gap)
  Resultado: COERENTE
  Saved: 11_gap_vs_efficiency.png


### 10.1 Sanity Check: Quais Hospitais Atendem os Piores e Melhores Municípios?

In [15]:
# Top 10 worst municipalities — which hospitals do their patients go to?
worst10_codes = city_agg.nlargest(10, "gap_composite")["MUNIC_RES"].values
best10_codes = city_agg.nsmallest(10, "gap_composite")["MUNIC_RES"].values

def hospital_profile(patient_codes, label):
    pts = df[df["MUNIC_RES"].isin(patient_codes)]
    hosp = pts.groupby("CNES").agg(
        n_patients=("DIAS_PERM", "count"),
        mean_los=("DIAS_PERM", "mean"),
        mortality=("MORTE", "mean"),
    ).reset_index()
    hosp = hosp.merge(hosp_eff[["CNES", "efficiency"]], on="CNES", how="left")
    hosp["name"] = hosp["CNES"].apply(lambda c: hospital_name(c, names_df))
    hosp["above_median"] = hosp["efficiency"] > eff_median
    hosp = hosp.sort_values("n_patients", ascending=False)

    print(f"\n=== {label} ===")
    print(f"Total pacientes: {hosp['n_patients'].sum():,}")
    print(f"Hospitais utilizados: {len(hosp)}")
    valid_h = hosp.dropna(subset=["efficiency"])
    if len(valid_h) > 0:
        mean_eff = np.average(valid_h["efficiency"], weights=valid_h["n_patients"])
        pct_above = valid_h.loc[valid_h["above_median"], "n_patients"].sum() / valid_h["n_patients"].sum()
        print(f"Efici\u00eancia m\u00e9dia ponderada: {mean_eff:.3f} (mediana sistema: {eff_median:.3f})")
        print(f"% pacientes em hospitais acima da mediana: {pct_above:.0%}")
    print(f"\nTop 10 hospitais:")
    print(hosp[["name", "n_patients", "mean_los", "mortality", "efficiency"]].head(10)
          .to_string(index=False, float_format="%.3f"))
    return hosp

worst_hosp = hospital_profile(worst10_codes, "HOSPITAIS DOS 10 PIORES MUNIC\u00cdPIOS")
best_hosp = hospital_profile(best10_codes, "HOSPITAIS DOS 10 MELHORES MUNIC\u00cdPIOS")

# Statistical comparison
w_eff = worst_hosp.dropna(subset=["efficiency"])
b_eff = best_hosp.dropna(subset=["efficiency"])
if len(w_eff) > 0 and len(b_eff) > 0:
    w_mean = np.average(w_eff["efficiency"], weights=w_eff["n_patients"])
    b_mean = np.average(b_eff["efficiency"], weights=b_eff["n_patients"])
    u_stat, u_p = stats.mannwhitneyu(
        w_eff["efficiency"].values, b_eff["efficiency"].values, alternative="less"
    )
    print(f"\n=== COMPARA\u00c7\u00c3O ===")
    print(f"Efici\u00eancia m\u00e9dia (piores munic\u00edpios):  {w_mean:.3f}")
    print(f"Efici\u00eancia m\u00e9dia (melhores munic\u00edpios): {b_mean:.3f}")
    print(f"Mann-Whitney U: p={u_p:.4f}")
    print(f"Resultado: {'VALIDADO' if u_p < 0.05 else 'N\u00c3O VALIDADO'} — "
          f"hospitais dos piores munic\u00edpios {'s\u00e3o' if u_p < 0.05 else 'n\u00e3o s\u00e3o'} "
          f"significativamente menos eficientes")


=== HOSPITAIS DOS 10 PIORES MUNICÍPIOS ===
Total pacientes: 878
Hospitais utilizados: 36
Eficiência média ponderada: 0.361 (mediana sistema: 0.564)
% pacientes em hospitais acima da mediana: 25%

Top 10 hospitais:
                                                      name  n_patients  mean_los  mortality  efficiency
                              CONJUNTO HOSPITALAR SOROCABA         479     4.635      0.021       0.222
                             HOSPITAL REGIONAL DE SOROCABA          85     1.976      0.000       0.596
HOSPITAL DAS CLINICAS DA FACULDADE DE MEDICINA DE BOTUCATU          53     6.019      0.000       0.309
                                        SANTA CASA DE TUPA          49     4.020      0.041       0.444
                              HOSPITAL AMARAL CARVALHO JAU          33     2.424      0.030       0.710
                                 HOSPITAL DE BASE DE BAURU          21     1.429      0.000       0.707
                                   HOSPITAL ESTADUAL BAUR

## 11. Veredicto da Validação

In [16]:
print("=" * 65)
print("  VEREDICTO DA VALIDA\u00c7\u00c3O DO MODELO")
print("=" * 65)

checks = [
    ("Calibra\u00e7\u00e3o (Brier < base rate)",
     brier_long < y_long.mean() * (1 - y_long.mean())),
    ("Estabilidade temporal (delta < 0.05)",
     abs(r2_t - r2) < 0.05 and abs(auc_long_t - auc_long) < 0.05),
    (f"Gaps significativos (>25% dos munic\u00edpios)",
     n_sig / len(city_agg) > 0.25),
    ("Vi\u00e9s subgrupo < 0.5 dias",
     max(abs(r["bias"]) for r in subgroup_results) < 0.5),
    ("Correla\u00e7\u00e3o gap vs efici\u00eancia (\u03c1 < 0, p < 0.05)",
     r_eff < 0 and p_eff < 0.05),
    ("Sanity check: piores munic. usam hospitais piores",
     w_mean < b_mean if "w_mean" in dir() and "b_mean" in dir() else False),
]

n_pass = 0
for name, passed in checks:
    status = "PASS" if passed else "FAIL"
    icon = "\u2705" if passed else "\u274c"
    print(f"  {icon} {status:4s} | {name}")
    n_pass += int(passed)

print(f"\n  Resultado: {n_pass}/{len(checks)} testes passaram")
if n_pass >= 5:
    print("  \u2192 MODELO VALIDADO para uso em an\u00e1lise de gaps municipais")
elif n_pass >= 3:
    print("  \u2192 MODELO PARCIALMENTE VALIDADO \u2014 usar com cautela")
else:
    print("  \u2192 MODELO N\u00c3O VALIDADO \u2014 n\u00e3o usar para decis\u00f5es")
print("=" * 65)

  VEREDICTO DA VALIDAÇÃO DO MODELO
  ✅ PASS | Calibração (Brier < base rate)
  ✅ PASS | Estabilidade temporal (delta < 0.05)
  ✅ PASS | Gaps significativos (>25% dos municípios)
  ✅ PASS | Viés subgrupo < 0.5 dias
  ✅ PASS | Correlação gap vs eficiência (ρ < 0, p < 0.05)
  ✅ PASS | Sanity check: piores munic. usam hospitais piores

  Resultado: 6/6 testes passaram
  → MODELO VALIDADO para uso em análise de gaps municipais


In [17]:
validation_metrics = {
    "total_patients": int(len(kidney)),
    "total_municipalities": int(kidney["MUNIC_RES"].nunique()),
    "municipalities_analyzed": int(len(city_agg)),
    "min_admissions_threshold": MIN_ADMISSIONS,
    "model_los_r2": round(r2, 3),
    "model_los_mae": round(mae, 2),
    "model_long_stay_auc": round(auc_long, 3),
    "model_mortality_auc": round(auc_mort, 3),
    "brier_long_stay": round(brier_long, 4),
    "brier_mortality": round(brier_mort, 4),
    "temporal_los_r2": round(r2_t, 3),
    "temporal_long_stay_auc": round(auc_long_t, 3),
    "temporal_mortality_auc": round(auc_mort_t, 3),
    "temporal_stable": bool(temporal_stable),
    "gaps_significant_count": int(n_sig),
    "gaps_significant_pct": round(n_sig / len(city_agg), 3),
    "gaps_sig_worse": int(n_sig_pos),
    "gaps_sig_better": int(n_sig_neg),
    "max_subgroup_bias": round(max_bias, 3),
    "gap_vs_efficiency_rho": round(r_eff, 3),
    "gap_vs_efficiency_p": round(p_eff, 4),
    "validation_checks_passed": n_pass,
    "validation_checks_total": len(checks),
    "worst_city": str(city_agg.iloc[0]["city_name"]),
    "worst_city_gap": round(float(city_agg.iloc[0]["gap_composite"]), 3),
    "best_city": str(city_agg.iloc[-1]["city_name"]),
    "best_city_gap": round(float(city_agg.iloc[-1]["gap_composite"]), 3),
    "features_used": feature_cols,
}

save_metrics(validation_metrics, "11_city_risk_model")

city_agg.to_csv(DATA_DIR / "city_risk_scorecard.csv", index=False)
print(f"\nSaved city scorecard with {len(city_agg)} municipalities")
print(f"Saved metrics with {len(validation_metrics)} fields")

  Saved metrics: 11_city_risk_model.json

Saved city scorecard with 489 municipalities
Saved metrics with 28 fields
